# GeoMech-LogML — Notebook 02: Training, Well-wise CV, Uncertainty & SHAP

End-to-end programmatic workflow mirroring the Streamlit app:

1. Generate / load data → clean → engineer features
2. Train Random Forest, XGBoost and a shallow MLP with **strictly well-wise CV**
3. Compare blind-well metrics; run the with/without-Vp **ablation**
4. Validate **prediction intervals** (conformal + QRF) on held-out wells
5. Explain predictions with **SHAP** (summary, dependence, waterfall)
6. Export a report

In [ ]:
from geomech_logml.data.synthetic import SyntheticConfig, generate_dataset
from geomech_logml.pipeline import ExperimentConfig, run_experiment, run_ablation, interval_coverage_summary

cfg_syn = SyntheticConfig(n_wells=8, step_m=1.0, seed=42)
data = generate_dataset(cfg_syn)
print(data.WELL.unique(), len(data), 'rows')

In [ ]:
%%time
exp_cfg = ExperimentConfig(
    feature_set='eng_with_vp',
    model_keys=['random_forest', 'xgboost', 'mlp'],
    cv_strategy='well_kfold', n_splits=4, alpha=0.10, seed=42,
    hyper_overrides={'mlp': {'max_iter': 1500, 'hidden_layer_sizes': (48, 24)}},
)
res = run_experiment(data, exp_cfg)
res.metrics

## 2. Blind-well performance — pooled out-of-fold predictions

In [ ]:
import pandas as pd
pivot = res.metrics.pivot(index='Model', columns='Target', values='R2').round(3)
pivot.style.highlight_max(axis=0)

## 3. Ablation: does the optional sonic log (Vp) help?

Compare the same models with and without Vp-derived features (`VP`, `PHIS`, `AI`).
The legacy-suite feature set remains usable when only triple-combo logs exist.

In [ ]:
%%time
abl = run_ablation(data, exp_cfg)
abl[['Model','Target','R2_without_Vp','R2_with_Vp','dR2_Vp','dRMSE_Vp']].round(4)

## 4. Prediction intervals — honest validation on held-out wells

In [ ]:
interval_coverage_summary(res).round(3)

*Conformal* intervals calibrate on residuals from wells unseen in training and therefore
absorb inter-well bias. *QRF* intervals adapt row-by-row to local data spread but do not
absorb systematic well shifts — the two are complementary.

## 5. SHAP explanations

In [ ]:
import matplotlib.pyplot as plt
from geomech_logml.interpretability.shap_explainer import ShapExplainer

X, Y, g = res.X_core, res.Y_core, res.groups
ex = ShapExplainer('random_forest', 'UCS', res.final_models['random_forest']['UCS'],
                   background=X.iloc[:300])
Xs = X.iloc[:200]
_ = ex.summary_figure(Xs); plt.show()

In [ ]:
top = ex.top_features(Xs, k=3)
print('top features:', top)
fig = ex.dependence_figure(Xs, top[0]); plt.show()

In [ ]:
# Per-depth explanation (waterfall) for one row
row = X.iloc[10]
fig = ex.waterfall_figure(row); plt.show()

## 6. Export predictions & report

In [ ]:
import os; os.makedirs('outputs', exist_ok=True)
res.curves.to_csv('outputs/nb_curve_predictions.csv', index=False)
from geomech_logml.app.report import build_report
md_report = build_report(res, 'synthetic Agbada-like (8 wells)')
open('outputs/nb_report.md', 'w').write(md_report)
print(md_report[:600])